# DDC Mission 1 · Frozen Wav2Vec2 · Colab CUDA
Colab은 CUDA embedding 추출을 담당하고, scaler/LR 학습·Validation 평가·결과 정리는 로컬에서 진행합니다. 기존 실험 정의는 유지합니다. **Run all도 benchmark 뒤 STOP**하며 전체 추출/accuracy 평가는 하지 않습니다.
먼저 Colab 런타임 유형을 GPU로 설정하세요. 각 셀을 순서대로 실행하세요.

현재 로컬 Wav2Vec2 코드가 Git 미추적 상태입니다. GitHub에 해당 파일이 없다면 clone 뒤 제공된 `colab_code_bundle.zip`을 업로드하세요. 자동 push는 수행하지 않습니다.

## 1. GitHub clone → mission1/eunkyo checkout

In [ ]:
import subprocess, sys
from pathlib import Path
REPO_URL = "https://github.com/eunkyo00/DDC2026-AIGO.git"
REPO_DIR = Path("/content/DDC2026-AIGO")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", "mission1/eunkyo", "--single-branch", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "mission1/eunkyo"], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "status", "--short", "--branch"], check=True)
COLAB_DIR = REPO_DIR / "mission-1/ssl/wav2vec2_frozen/colab"
# Private repository: Colab에서 본인 계정으로 인증한 뒤 다시 실행하세요. 토큰을 notebook에 저장하지 마세요.

## 2. 준비 파일 확인 / 로컬 코드 bundle 업로드
최초 전달된 zip을 사용하면 미추적 코드도 실행할 수 있습니다. zip에는 WAV와 대용량 CSV가 없습니다.

In [ ]:
USE_LOCAL_BUNDLE = True  # 이 작업 결과 zip 사용. GitHub에 모두 반영했다면 False로 변경
if USE_LOCAL_BUNDLE:
    from google.colab import files
    import zipfile, io
    uploaded = files.upload()  # colab_code_bundle.zip 하나 선택
    if len(uploaded) != 1:
        raise RuntimeError("코드 bundle zip 하나를 선택하세요")
    with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
        for name in archive.namelist():
            dest = (REPO_DIR / name).resolve()
            if not dest.is_relative_to(REPO_DIR.resolve()):
                raise RuntimeError("Unsafe archive path")
        archive.extractall(REPO_DIR)
required = [COLAB_DIR / "colab_runner.py", COLAB_DIR / "requirements.txt",
            COLAB_DIR.parent / "run_wav2vec2_frozen.py",
            COLAB_DIR.parent / "results/smoke_calls.csv",
            COLAB_DIR.parent / "results/benchmark_calls.csv",
            COLAB_DIR.parent / "results/benchmark_results.json"]
for path in required:
    if not path.is_file():
        raise FileNotFoundError(f"필수 코드/fixture 없음: {path}")
print("준비 파일 확인 완료")

## 3. Dependencies 설치
Colab에 설치된 CUDA PyTorch는 유지합니다. 나머지는 기존 실험 버전으로 고정합니다. 설치 실패 시 임의로 버전을 풀지 말고 오류를 전달하세요. 설치 전 import된 NumPy 버전이 남아 있으면 Python 세션을 재시작한 뒤 계속하세요.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(COLAB_DIR / "requirements.txt")], check=True)
subprocess.run([sys.executable, "-c", "import torch; print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)"], check=True)

## 4. Google Drive mount 및 실제 경로 탐색
공유받은 폴더가 보이지 않으면 Drive에서 내 드라이브에 바로가기를 추가한 뒤 경로를 확인하세요. Shared drives가 제공되는 계정은 mount 아래 표시되는 경로를 사용하세요. 아래 셀은 지정 폴더의 바로 아래 항목만 표시합니다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
BROWSE_DIR = Path("/content/drive")  # 실제 표시된 폴더로 변경하며 다시 실행
if not BROWSE_DIR.is_dir():
    raise FileNotFoundError(BROWSE_DIR)
for path in sorted(BROWSE_DIR.iterdir())[:100]:
    print("DIR " if path.is_dir() else "FILE", path)

## 5. 경로 입력
`DATA_ROOT`는 manifest의 `wav_relative_path`를 붙일 루트입니다(예: 그 아래 `Training/`이 있음). 경로를 추측하지 말고 위 셀로 확인하세요. OUTPUT_DIR는 쓰기 가능한 본인 Drive의 이 CUDA 실행 전용 폴더로 설정하세요. Mac cache와 섞지 마세요.

In [ ]:
import json
DATA_ROOT = ""      # mount한 Drive의 원본 WAV 루트
SEGMENTS_CSV = ""   # 기존 eda/eda_outputs/segments.csv의 Drive 사본
MANIFEST_CSV = ""   # 기존 validation/manifests/calls.csv의 Drive 사본
SPLIT_CSV = str(REPO_DIR / "mission-1/validation/split_assignments.csv")  # 기존 fixed split
OUTPUT_DIR = ""     # mount한 Drive의 쓰기 가능한 결과 폴더
config = {k: globals()[k] for k in ("DATA_ROOT", "SEGMENTS_CSV", "MANIFEST_CSV", "SPLIT_CSV", "OUTPUT_DIR")}
for key, value in config.items():
    if not value:
        raise ValueError(f"실제 경로를 입력하세요: {key}")
for key in ("SEGMENTS_CSV", "MANIFEST_CSV", "SPLIT_CSV"):
    if not Path(config[key]).is_file():
        raise FileNotFoundError(config[key])
CONFIG_PATH = Path("/content/wav2vec2_colab_config.json")
CONFIG_PATH.write_text(json.dumps(config, ensure_ascii=False, indent=2))
# manifest 절대 Mac 경로는 사용하지 않음. relative_path만 Drive 루트에 연결.
import csv
with Path(MANIFEST_CSV).open(encoding="utf-8-sig") as f:
    first = next(csv.DictReader(f))
print("첫 WAV 연결 예:", Path(DATA_ROOT) / first["wav_relative_path"])
sys.path.insert(0, str(COLAB_DIR))
from colab_session import ColabSession
session = None
def run_stage(stage):
    global session
    # 허용 단계는 네 개뿐: Run all로 extract를 실행할 수 없음.
    if stage not in {"cuda", "preflight", "smoke", "benchmark"}:
        raise ValueError("Notebook stops after benchmark")
    if session is None:
        session = ColabSession(CONFIG_PATH)
    if stage != "cuda":
        return getattr(session, stage)()

## 6. CUDA/GPU 확인
CUDA unavailable이면 즉시 중단합니다. 실제 model device는 smoke에서 checkpoint 로드 직후 추가 출력합니다.

In [ ]:
run_stage("cuda")

## 7. 최소 Drive preflight + 기존 CSV 연결 검증
27,985 calls / Train 22,388 / Validation 5,597 / caller segments 442,639와 split SHA256를 확인합니다. WAV 읽기는 기존 smoke/benchmark 대상의 합집합으로 제한합니다. 해당 WAV를 65,536 sample 블록으로 open/read하고 8kHz mono·구간 경계·missing WAV·I/O error를 검사합니다. 나머지 WAV는 향후 전체 추출 중 검사하므로 여기서 전체 WAV 무오류를 주장하지 않습니다. GPU forward는 없습니다.

In [ ]:
run_stage("preflight")

## 8. 기존 실제 6-call smoke
원래 저장된 call ID 사용. `(6, 768)`, finite, frozen, 실제 CUDA device, FP32 검사. classifier는 실행하지 않습니다. 전체 CUDA embedding을 받은 뒤 로컬에서 기존 Train-only scaler/LR을 실행합니다.

In [ ]:
run_stage("smoke")

## 9. 기존 실제 20-call benchmark → 예상 시간
smoke 통과 기록과 입력/code/version fingerprint가 일치할 때만 실행됩니다. CUDA 메모리는 allocated/reserved peak, RAM은 subprocess 수명 동안의 peak RSS입니다. runtime은 기존 Mac과 같이 WAV read + extraction이며 model load는 제외하며 classifier는 실행하지 않습니다.

In [ ]:
run_stage("benchmark")
result = json.loads((Path(OUTPUT_DIR) / "benchmark.json").read_text())
mac = result["mac_reference"]["measurement"]
gpu = result["measurement"]
print("항목 | Mac MPS | Colab CUDA")
for key in ("calls", "segments", "caller_audio_seconds", "elapsed_seconds", "seconds_per_call", "seconds_per_segment", "audio_realtime_factor"):
    print(f"{key} | {mac[key]} | {gpu[key]}")
print("Mac audio-scaled 예상 시간:", result["mac_reference"]["projection"]["projected_full_hours"], "hours")
print("Colab 예상 시간:", result["projection"])
print("STOP — full extraction은 시작하지 않았습니다.")

## STOP — 이번 실행은 여기까지
`OUTPUT_DIR`의 `preflight.json`, `smoke.json`, `benchmark.json`과 오류가 있다면 `failures/`를 다음 대화에 전달하세요.

전체 추출 실행 셀은 없습니다. Benchmark 결과 검토와 사용자의 명시적 승인 후에만 별도 실행합니다. 미래의 별도 CLI는 `--stage extract`와 `--confirm-full-extraction`을 모두 명시해야 실행됩니다. Drive call cache/resume 구조와 세션 종료 후 복구 방법은 README를 참고하세요. **이번에는 실행하지 마세요.**